# The stocks funds started buying in size: new $100M positions in Q2 2026 13Fs

Every position worth $100 million or more that a fund manager didn't hold the quarter before, from 13F filings for the quarter ended June 30, 2026: which stocks drew the most new money and the largest single bets.

Read it with charts at https://www.quantisnow.com/sec-api/notebooks/new-fund-positions-q2-2026. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

A 13F shows what a manager held at quarter end. Compared with the quarter before, it also shows what they started holding. A brand-new position worth $100 million or more is a decision, not a rounding error, so this notebook keeps only those.

The helper pages through a query 50 records at a time, one API call per page. Every notebook here starts with it.

In [1]:
import os, requests
import pandas as pd

API = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(dataset, query, start=0, size=50, sort=None):
    body = {"query": query, "from": start, "size": size}
    if sort:
        body["sort"] = sort
    r = requests.post(f"{API}/{dataset}", json=body,
                      headers={"Authorization": f"Bearer {KEY}"})
    r.raise_for_status()
    return r.json()

def search_all(dataset, query, sort=None):
    """Every record that matches, 50 per call."""
    rows, start = [], 0
    while True:
        page = search(dataset, query, start, 50, sort)
        rows += page["data"]
        start += 50
        if start >= min(page["total"]["value"], 10_000):
            return rows

## Pull the new positions

In [2]:
new = pd.json_normalize(search_all("13f/holdings",
    "period:2026-06-30 AND change:new AND valueUsd:>=100000000"))
print(f"{len(new):,} new positions of $100M or more, opened by {new['manager.cik'].nunique()} managers, "
      f"${new['valueUsd'].sum() / 1e9:.1f}B in all")

1,705 new positions of $100M or more, opened by 550 managers, $1019.4B in all

## Which stocks drew the most managers

In [3]:
popular = (new.groupby(["issuer.ticker", "issuer.name"])
    .agg(funds=("manager.cik", "nunique"), usd=("valueUsd", "sum"))
    .sort_values(["funds", "usd"], ascending=False))
popular["funds"].head(15).droplevel(1).plot.barh(title="Stocks with the most new $100M+ fund positions, Q2 2026")

Stocks with the most new $100M+ fund positions, Q2 2026
SPCX	142
HONA	65
HON	64
DD	31
FDXF	29
CBRS	25
TRI	23
AMD	21
INTC	19
SPY	18
MU	17
MSFT	16
SNDK	15
AMZN	15
MAIR	15

In [4]:
popular.head(15)

issuer.ticker,issuer.name,funds,usd
SPCX,Space Exploration Technologies Corp.,142,"528,599,971,420"
HONA,Honeywell Aerospace Inc.,65,"41,223,586,110"
HON,Honeywell International Inc.,64,"43,794,835,357"
DD,DuPont de Nemours Inc.,31,"8,705,108,111"
FDXF,FedEx Freight Holding Company Inc.,29,"9,761,007,046"
CBRS,Cerebras Systems Inc.,25,"12,468,649,942"
TRI,Thomson Reuters Corporation,23,"6,377,495,322"
AMD,Advanced Micro Devices Inc.,21,"4,826,524,315"
INTC,Intel Corporation,19,"5,919,250,308"
SPY,State Street SPDR S&P 500 ETF Trust Unit,18,"10,354,219,482"


The list says less about stock picking than about what came new to market. SpaceX listed in the quarter, so every fund that held it on June 30 holds a new position: 142 managers, $529 billion. Honeywell Aerospace, spun off from Honeywell, comes next with 65, and Honeywell itself with 64, then DuPont and the FedEx Freight spin-off. Further down, among stocks that have traded for years, are Thomson Reuters, AMD and Intel, with 23, 21 and 19 new $100 million positions.

## The largest single new positions

In [5]:
new.nlargest(12, "valueUsd")[["manager.name", "issuer.ticker", "valueUsd"]]

manager.name,issuer.ticker,valueUsd
Alphabet Inc.,SPCX,"94,176,237,970"
Valor Management LLC,SPCX,"86,013,406,596"
FMR LLC,SPCX,"51,655,459,728"
PUBLIC INVESTMENT FUND,SPCX,"26,337,528,228"
BAMCO INC /NY/,SPCX,"24,907,141,616"
D1 Capital Partners L.P.,SPCX,"21,535,575,760"
NVIDIA CORP,SPCX,"20,975,594,582"
"SC US (TTGP), LTD.",SPCX,"20,933,621,114"
Capricorn Investment Group LLC,SPCX,"20,640,198,965"
Darsana Capital Partners LP,SPCX,"17,335,782,797"


Eleven of the twelve largest new positions are SpaceX, led by Alphabet at $94 billion, a stake it held before the listing. The only other one is SG Americas Securities with $12.5 billion of Nvidia, which for a bank's securities arm is more likely hedging or client flow than a bet.

## Take it further

To leave out new listings and spin-offs, drop tickers whose `previousShares` is missing for every manager, or check the issuer's filings for an S-1 or Form 10 in the quarter. `change:sold` gives the positions funds closed instead. The [13F dataset page](https://www.quantisnow.com/sec-api/13f-holdings) lists every field.